# Frames in a notebook with a plain iframe (no anywidget)

No packages to install: this uses only the standard library and
`IPython.display.IFrame`, which every Jupyter already has. It works anywhere
that renders HTML output: JupyterLab, classic Notebook, VS Code, Colab,
nbviewer, and exported HTML.

**The tradeoff:** data goes **one way, once.** The inputs are encoded into the
frame's URL (the `inputs` hash param), so:

- to change the inputs, **change them and re-run the cell**. The iframe is
  rebuilt from scratch and the frame restarts with the new inputs. Nothing
  updates in place.
- nothing comes back. Outputs, `on_outputs_change` and `pipe_to` need the
  `metaframe-widget` package (see `demo.ipynb`).

## The helper

`inputs` is encoded exactly like every JSON hash param in framejs
(`@metapages/hash-query`): `btoa(encodeURIComponent(JSON.stringify(value)))`.

In [ ]:
import base64
import json
import urllib.parse

from IPython.display import IFrame


def encode_json_hash_param(value) -> str:
    """Encode `value` like @metapages/hash-query encodes a JSON hash param:
    btoa(encodeURIComponent(JSON.stringify(value)))."""
    text = json.dumps(value, separators=(",", ":"), ensure_ascii=False)
    # The characters encodeURIComponent leaves alone.
    uri_component = urllib.parse.quote(text, safe="-_.!~*'()")
    return base64.b64encode(uri_component.encode("ascii")).decode("ascii")


def frame_url_with_inputs(url: str, inputs: dict) -> str:
    """`url` with its `inputs` hash param set, keeping any other hash params."""
    base, _, fragment = url.partition("#")
    params = [
        p for p in fragment.lstrip("?").split("&")
        if p and not p.startswith("inputs=")
    ]
    params.append("inputs=" + encode_json_hash_param(inputs))
    return base + "#?" + "&".join(params)


def frame_with_inputs(url: str, inputs: dict, width="100%", height=300) -> IFrame:
    """A frame as a plain iframe, fed `inputs`. Re-run the cell to change them."""
    return IFrame(frame_url_with_inputs(url, inputs), width=width, height=height)

## 1. Send inputs to a frame

An "echo" frame that displays whatever inputs it receives. A
`https://framejs.io/j/<id>` URL and the same `https://framejs.app/j/<id>` work
the same way.

In [ ]:
ECHO = "https://framejs.io/j/019f61392e0778f8aba8dd7ffe35a83c"

frame_with_inputs(ECHO, {"message": "hello from Python", "data": [1, 2, 3]}, height=150)

## 2. Change the inputs: re-run the cell

Edit `message` and re-run (`Shift+Enter`). The old iframe is thrown away and a
new one starts with the new inputs.

In [ ]:
message = "change me, then re-run this cell"

frame_with_inputs(ECHO, {"message": message}, height=150)

## 3. Plot data computed in Python

A Plotly frame that draws whatever series it is given. The data travels in the
URL, so keep it to kilobytes. For anything big, pass a URL instead:
`{"type": "url", "value": "https://..."}` is fetched by the frame itself.

In [ ]:
import math

x = [i * 4 * math.pi / 200 for i in range(201)]
data = {
    "x": x,
    "series": {
        "sin(x)": [math.sin(v) for v in x],
        "0.5 sin(3x)": [0.5 * math.sin(3 * v) for v in x],
    },
}

frame_with_inputs(
    "https://framejs.io/j/f500b0bdbe8d4a1690803014bc9ac918",
    {"data": data},
    height=420,
)